In [2]:
import os
import sys
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import re
import anndata as ad
import statistics
import torch
import scvi
import tempfile
import sklearn
import mudata as md
import muon as mu
md.set_options(pull_on_update=False)
import muon
from datetime import datetime
from scib_metrics.benchmark import Benchmarker
from sklearn_ann.kneighbors.annoy import AnnoyTransformer
from multiprocessing import Pool
import uuid
print("Last run with scvi-tools version:", scvi.__version__)
scvi.settings.num_threads = 16
scvi.settings.seed = 0
sc._settings.n_jobs=16
sc.settings.verbosity = 4
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(8,8), format='png')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 0


Last run with scvi-tools version: 1.3.3


In [3]:
sc.logging.print_header()

/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/session_info2/__init__.py:125: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  and (v := getattr(pkg, "__version__", None))
/tmp/ipykernel_1952570/1023403583.py:1: RuntimeWarning: Failed to import dependencies for application/vnd.jupyter.widget-view+json representation. (ModuleNotFoundError: No module named 'ipywidgets')
  sc.logging.print_header()


Package,Version
ipykernel,6.30.1
scipy,1.15.3
numpy,2.2.4
pandas,2.3.1
scanpy,1.11.4
seaborn,0.13.2
matplotlib,3.10.5
anndata,0.12.1
torch,2.8.0 (2.8.0+cu128)
scvi-tools,1.3.3


In [4]:
Classification = sys.argv[1]
dataset = sys.argv[2]

In [5]:
Classification='Classification_L3'
dataset='NDNs'

In [6]:
obj_path = f'/home/liyanguo/MyImmuCell/06_Finnal_raw_count/{Classification}/{dataset}/'
sc.settings.figdir = obj_path

In [7]:
adata = sc.read_h5ad(f"{obj_path}{dataset}_scRNA_count_HVG.h5ad")

In [ ]:
adt = sc.read_h5ad(f"{obj_path}{dataset}_scADT_count.h5ad")

In [ ]:
indices = pd.read_csv(f"{obj_path}/{dataset}_indices_labels.csv",index_col=0)

In [ ]:
indices = indices[['Classification_L4','Classification_L3','Classification_L2','Classification_L1','leiden_cluster']]

In [ ]:
if (adata.obs.index == indices.index).all():
    adata.obs = adata.obs.join(indices,how='left')
    adt.obs = adt.obs.join(indices, how='left')
else:
    raise ValueError('Indices error.')

# 1. Load latent representation and umap

In [ ]:
adata.obsm["X_TOTALVI"]=np.load(f"{obj_path}{dataset}_X_TOTALVI.npy")

In [ ]:
print(f"Do sc.pp.neighbors with AnnoyTransformer,{datetime.now()}")
sc.pp.neighbors(adata, transformer=AnnoyTransformer(20), use_rep='X_TOTALVI')

In [ ]:
print(f"Run UMAP,{datetime.now()}")
sc.tl.umap(adata, min_dist=0.5)

In [ ]:
#Not save
#adata.write(f"{obj_path}{dataset}_scRNA_count_HVG.h5ad",compression="gzip")

In [ ]:
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0]  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1]  #

In [ ]:
indices.head()

In [ ]:
indices.to_csv(f"{obj_path}/{dataset}_indices_labels_umap.csv")

# 2. Plot leiden umap

In [ ]:
groupby='Classification_L4'

In [ ]:
print(f"Plot umap,{datetime.now()}")
sc.pl.umap(
    adata,
    color=groupby,
    legend_fontsize=6,legend_loc='on data',ncols=3,frameon=False,
    save=f'_{dataset}_{groupby}',show=False
)

# 3. Process adt

In [ ]:
adt.obsm = adata.obsm

In [ ]:
adt.X.max()

In [ ]:
adt.layers["clr"] = mu.prot.pp.clr(adt,inplace= False).X.copy()

In [ ]:
adt.write(f"{obj_path}{dataset}_scADT_count.h5ad",compression="gzip")

In [ ]:
import session_info
session_info.show()

In [ ]:
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L3 NKT &
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L3 DN_T_cells &
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L3 CD4pos_Treg &
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L3 Memory_B_cells &
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L3 Vδ2pos_T_cells &
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L3 CD8pos_Tem &
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L3 Help_memory_T_cells &

nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L2 CD4pos_T_cells & 
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L2 Non_MAIT_NKT_CD8pos_T_cells &
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L2 ILCs &
nohup python 27D_CITEseq_TOTALVI_umap_subset.py Classification_L2 B_cells &